# RiftLab CC-703: treino no Colab (GPU)

Este notebook roda as três execuções do protocolo no dataset v4: compra, planejamento e contexto rico. Em cada uma, a seleção usa só a validação e o teste roda **uma única vez**. O código é o mesmo do projeto (`riftlab`).

**Antes de começar**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.
2. No seu Google Drive, crie a pasta `riftlab-cc703` e coloque nela o arquivo `riftlab-cc703-colab.zip`.
3. Rode as células em ordem.

**Privacidade**
- Os datasets do pacote não têm id de partida: foram trocados por índices opacos.
- Não há nome nem id de jogador.
- Os catálogos são públicos (Data Dragon).

**Se a sessão cair**
- Rode tudo de novo.
- Execuções já concluídas são puladas, e os resultados ficam no Drive (`riftlab-cc703/out`).
- Uma execução interrompida no meio do treino aparece como erro, com instrução para renomear a pasta e recomeçar só ela.

**No fim:** a última célula gera `riftlab-cc703-resultados.zip` no Drive. Mande esse arquivo para o Claude.

In [ ]:
# 1. GPU disponível?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Drive e pacote
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, zipfile, pathlib
BASE = pathlib.Path('/content/drive/MyDrive/riftlab-cc703')
ZIP = BASE / 'riftlab-cc703-colab.zip'
assert ZIP.exists(), f'coloque o pacote em {ZIP}'
BUNDLE = pathlib.Path('/content/bundle')
if not BUNDLE.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(BUNDLE)
manifest = json.loads((BUNDLE / 'MANIFEST.json').read_text())
bad = [n for n, h in manifest['files'].items() if hashlib.sha256((BUNDLE / n).read_bytes()).hexdigest() != h]
assert not bad, f'arquivos alterados: {bad[:5]}'
print('pacote íntegro:', len(manifest['files']), 'arquivos; SHA-256 do zip:', hashlib.sha256(ZIP.read_bytes()).hexdigest()[:16])

In [ ]:
# 3. Mesmas versões usadas no projeto (conferidas num processo novo, sem misturar com o numpy já carregado)
!pip -q install catboost==1.2.10 numpy==2.5.3
!python -c "import catboost, numpy; print('catboost', catboost.__version__, '| numpy', numpy.__version__)"


In [ ]:
# 4. Cronometragem: a mesma configuração fixa (300 iterações) em GPU e em CPU
import os, subprocess, json
env = dict(os.environ, PYTHONPATH=str(BUNDLE / 'src'))
args = [str(BUNDLE / 'data/dataset-v4-features.json'), str(BUNDLE / 'catalog/champion-16.19.1-en_US.json'), str(BUNDLE / 'catalog/item-16.19.1-en_US.json')]
bench = []
for task, threads in (('GPU', 2), ('CPU', os.cpu_count())):
    proc = subprocess.run(['python', str(BUNDLE / 'bench_train.py'), *args, task, str(threads)], env=env, capture_output=True, text=True)
    print(proc.stdout or proc.stderr[-1500:])
    if proc.returncode == 0:
        bench.append(json.loads(proc.stdout))
(BASE / 'out').mkdir(parents=True, exist_ok=True)
(BASE / 'out' / 'bench-colab.json').write_text(json.dumps(bench, indent=1))
DEVICE = 'GPU' if any(b['taskType'] == 'GPU' for b in bench) else 'CPU'
print('dispositivo escolhido para o treino:', DEVICE)

In [ ]:
# 5. Treino e teste único das três execuções (pode levar algumas horas; resultados vão para o Drive)
threads = 2 if DEVICE == 'GPU' else os.cpu_count()
!PYTHONPATH={BUNDLE}/src python {BUNDLE}/train_colab.py {BUNDLE} {BASE}/out {DEVICE} {threads}

In [ ]:
# 6. Pacote de resultados para devolver ao Claude
import shutil
archive = shutil.make_archive(str(BASE / 'riftlab-cc703-resultados'), 'zip', str(BASE / 'out'))
print('gerado:', archive)
print(open(BASE / 'out' / 'timings.jsonl').read() if (BASE / 'out' / 'timings.jsonl').exists() else 'sem timings')